# Рекомендательная система фильмов

В ноутбуке загружаются фильмы и оценки пользователей, выполняется очистка данных и создаются четыре варианта рекомендаций: популярные фильмы, коллаборативные рекомендации по пользователям, похожие фильмы по оценкам и рекомендации по жанрам.

In [1]:
import os
import zipfile
import urllib.request
import pandas as pd
import numpy as np
%pip install deep-translator
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MultiLabelBinarizer
from scipy.sparse import csr_matrix

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## 1. Загрузка датасета

In [2]:
data_folder = "ml-latest-small"

movies = pd.read_csv(os.path.join(data_folder, "movies.csv"))
ratings = pd.read_csv(os.path.join(data_folder, "ratings.csv"))

print("Фильмы:", movies.shape)
print("Оценки:", ratings.shape)

Фильмы: (9742, 3)
Оценки: (100836, 4)


In [3]:
display(movies.head())

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
display(ratings.head())

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [ ]:
from deep_translator import GoogleTranslator
import time

translator = GoogleTranslator(source="en", target="ru")
translations = {}

def translate_title(title):
    if pd.isna(title):
        return title

    if title in translations:
        return translations[title]

    try:
        if title.endswith(")") and "(" in title:
            name = title[:title.rfind("(")].strip()
            year = title[title.rfind("("):]
        else:
            name = title
            year = ""

        translated = translator.translate(name)
        result = (translated + " " + year).strip()

        translations[title] = result
        time.sleep(0.1)
        return result

    except Exception:
        translations[title] = title
        return title

movies["title_original"] = movies["title"]

movies["title_ru"] = movies["title"].apply(translate_title)

def search_movie(title):
    title = title.strip().lower()

    if not title:
        return movies.iloc[0:0]

    result = movies[
        movies["title_ru"].str.lower().str.contains(title, na=False, regex=False)
        | movies["title_original"].str.lower().str.contains(title, na=False, regex=False)
    ]

    return result[["movieId", "title_ru", "title_original", "genres_ru"]]

display(search_movie("история игрушек"))


## 2. Очистка данных

In [ ]:
movies = movies.drop_duplicates(subset="movieId")
ratings = ratings.drop_duplicates(subset=["userId", "movieId"])
movies = movies.dropna(subset=["movieId", "title", "genres"])
ratings = ratings.dropna(subset=["userId", "movieId", "rating"])

In [ ]:
movies["movieId"] = movies["movieId"].astype(int)
ratings["movieId"] = ratings["movieId"].astype(int)
ratings["userId"] = ratings["userId"].astype(int)

In [ ]:
ratings = ratings[ratings["rating"].between(0.5, 5.0)]
ratings = ratings[ratings["movieId"].isin(movies["movieId"])]

print("Фильмов после очистки:", len(movies))
print("Оценок после очистки:", len(ratings))

## 3. Перевод жанров

In [ ]:
genre_translation = {
    "Action": "Боевик",
    "Adventure": "Приключения",
    "Animation": "Мультфильм",
    "Children": "Детский",
    "Comedy": "Комедия",
    "Crime": "Криминал",
    "Documentary": "Документальный",
    "Drama": "Драма",
    "Fantasy": "Фэнтези",
    "Film-Noir": "Нуар",
    "Horror": "Ужасы",
    "Musical": "Мюзикл",
    "Mystery": "Детектив",
    "Romance": "Мелодрама",
    "Sci-Fi": "Фантастика",
    "Thriller": "Триллер",
    "War": "Военный",
    "Western": "Вестерн",
    "(no genres listed)": "Жанр не указан"
}

def translate_genres(genres):
    return ", ".join(genre_translation.get(genre, genre) for genre in genres.split("|"))

movies["genres_ru"] = movies["genres"].apply(translate_genres)
display(movies[["title", "genres_ru"]].head(10))

## 4. Топ-10 популярных фильмов

In [ ]:
popular_movies = ratings.groupby("movieId").agg(
    average_rating=("rating", "mean"),
    ratings_count=("rating", "count")
).reset_index()

popular_movies = popular_movies[popular_movies["ratings_count"] >= 20]
popular_movies = popular_movies.merge(movies, on="movieId")
popular_movies = popular_movies.sort_values(
    ["average_rating", "ratings_count"], ascending=False
)

top_10_movies = popular_movies[
    ["title", "genres_ru", "average_rating", "ratings_count"]
].head(10)

display(top_10_movies.round(2))

## 5. Матрица «пользователь — фильм»

In [ ]:
user_movie = ratings.pivot_table(
    index="userId", columns="movieId", values="rating", fill_value=0
)

movie_user = user_movie.T

print("Матрица пользователей и фильмов:", user_movie.shape)
print("Матрица фильмов и пользователей:", movie_user.shape)

## 6. Коллаборативная фильтрация по пользователям

In [ ]:
user_similarity = cosine_similarity(user_movie)
user_similarity = pd.DataFrame(
    user_similarity,
    index=user_movie.index,
    columns=user_movie.index
)

def recommend_for_user(user_id, count=10):
    if user_id not in user_movie.index:
        return pd.DataFrame(columns=["title", "genres_ru", "score"])

    similar_users = user_similarity[user_id].drop(user_id).sort_values(ascending=False).head(10)
    user_ratings = user_movie.loc[user_id]
    unseen_movies = user_ratings[user_ratings == 0].index

    scores = {}
    for movie_id in unseen_movies:
        total_score = 0
        total_similarity = 0
        for other_user, similarity in similar_users.items():
            rating = user_movie.loc[other_user, movie_id]
            if rating > 0 and similarity > 0:
                total_score += rating * similarity
                total_similarity += similarity
        if total_similarity > 0:
            scores[movie_id] = total_score / total_similarity

    if not scores:
        return pd.DataFrame(columns=["title", "genres_ru", "score"])

    result = pd.DataFrame({
        "movieId": list(scores.keys()),
        "score": list(scores.values())
    })
    result = result.merge(movies, on="movieId")
    result = result.sort_values("score", ascending=False)
    return result[["title", "genres_ru", "score"]].head(count).reset_index(drop=True)

## 7. Похожие фильмы по оценкам пользователей

In [ ]:
movie_ids = movie_user.index.tolist()
movie_model = NearestNeighbors(metric="cosine", algorithm="brute")
movie_model.fit(csr_matrix(movie_user.values))

movie_to_row = {movie_id: i for i, movie_id in enumerate(movie_ids)}

def find_similar_movies(movie_title, count=10):
    found = movies[movies["title"].str.contains(movie_title, case=False, na=False)]

    if found.empty:
        return pd.DataFrame(columns=["title", "genres_ru", "similarity"])

    movie_id = int(found.iloc[0]["movieId"])
    if movie_id not in movie_to_row:
        return pd.DataFrame(columns=["title", "genres_ru", "similarity"])

    row_number = movie_to_row[movie_id]
    distances, indexes = movie_model.kneighbors(
        csr_matrix(movie_user.loc[movie_id].values.reshape(1, -1)),
        n_neighbors=min(count + 1, len(movie_ids))
    )

    result = []
    for distance, index in zip(distances[0], indexes[0]):
        similar_id = movie_ids[index]
        if similar_id != movie_id:
            result.append({
                "movieId": similar_id,
                "similarity": round(1 - float(distance), 3)
            })

    result = pd.DataFrame(result)
    result = result.merge(movies, on="movieId")
    return result[["title", "genres_ru", "similarity"]].head(count).reset_index(drop=True)

## 8. Рекомендации по содержанию фильма

In [ ]:
movies["genre_list"] = movies["genres"].apply(
    lambda value: [] if value == "(no genres listed)" else value.split("|")
)

genre_encoder = MultiLabelBinarizer()
genre_matrix = genre_encoder.fit_transform(movies["genre_list"])

content_model = NearestNeighbors(metric="cosine", algorithm="brute")
content_model.fit(genre_matrix)

movie_row_by_id = {movie_id: i for i, movie_id in enumerate(movies["movieId"])}

def recommend_by_content(movie_title, count=10):
    found = movies[movies["title"].str.contains(movie_title, case=False, na=False)]

    if found.empty:
        return pd.DataFrame(columns=["title", "genres_ru", "similarity"])

    movie_id = int(found.iloc[0]["movieId"])
    row_number = movie_row_by_id[movie_id]
    distances, indexes = content_model.kneighbors(
        genre_matrix[row_number].reshape(1, -1),
        n_neighbors=min(count + 1, len(movies))
    )

    result = []
    for distance, index in zip(distances[0], indexes[0]):
        similar_id = int(movies.iloc[index]["movieId"])
        if similar_id != movie_id:
            result.append({
                "movieId": similar_id,
                "similarity": round(1 - float(distance), 3)
            })

    result = pd.DataFrame(result)
    result = result.merge(movies, on="movieId")
    return result[["title", "genres_ru", "similarity"]].head(count).reset_index(drop=True)

## 9. Проверка рекомендаций

In [ ]:
display(find_similar_movies("Toy Story", 10))
display(recommend_by_content("Toy Story", 10))
display(recommend_for_user(int(ratings["userId"].iloc[0]), 10))

## 10. Сохранение файлов для API

In [ ]:
movies.to_csv("movies.csv", index=False, encoding="utf-8-sig")
ratings.to_csv("ratings.csv", index=False, encoding="utf-8-sig")
top_10_movies.to_csv("top_10_movies.csv", index=False, encoding="utf-8-sig")

print("Сохранены movies.csv, ratings.csv и top_10_movies.csv")

## 11. Итоговая проверка

In [ ]:
print("Фильмов:", len(movies))
print("Оценок:", len(ratings))
print("Пользователей:", ratings["userId"].nunique())
print("Топ-10 популярных фильмов:")
display(top_10_movies.round(2))